# The Concept

The Bridge Pattern splits a large class hierarchy into two separate hierarchies:
- Abstraction: The "Control" layer (The Remote).
- Implementation: The "Device" layer (The TV or Radio).

**Why?** Without the Bridge, if you have 2 types of remotes **(Basic, Advanced)** and 2 types of devices (TV, Radio), you might end up with 4 classes: `BasicTvRemote, BasicRadioRemote, AdvancedTvRemote, AdvancedRadioRemote`. This grows exponentially (Cartesian Product complexity). With the Bridge, the Remote simply holds a reference to a Device. You can mix and match them freely.

## The Classic OOP Way (Java-Style)

In this approach, we use strict Interfaces and Abstract Classes. This is explicit and rigid.

#### IMPLEMENTATION LAYER ( The Device )

In [1]:
from abc import ABC, abstractmethod

class Device(ABC):
    @abstractmethod
    def is_enabled(self) -> bool:pass
    
    @abstractmethod
    def enable(self): pass

    @abstractmethod
    def disable(self): pass

    @abstractmethod
    def set_volume(self, percent: int): pass

#### Concrete Implementations

In [3]:
class TV(Device):
    def __init__(self):
        self.on = False
        self.volume = 30

    def is_enabled(self) -> bool: return self.on
    def enable(self): 
        self.on = True
        print("TV: Turned ON")
        
    def disable(self): 
        self.on = False
        print("TV: Turned OFF")
        
    def set_volume(self, percent: int):
        self.volume = percent
        print(f"TV: Volume set to {self.volume}")

class Radio(Device):
    def __init__(self):
        self.on = False
        self.volume = 10

    def is_enabled(self) -> bool: return self.on
    def enable(self): 
        self.on = True
        print("Radio: Turned ON")
        
    def disable(self): 
        self.on = False
        print("Radio: Turned OFF")
        
    def set_volume(self, percent: int):
        self.volume = percent
        print(f"Radio: Volume set to {self.volume}")

#### ABSTRACTION LAYER ( The Remote )

In [4]:
class RemoteControl:
    """
    The Abstraction. It holds a reference (the 'bridge') to the Device.
    It delegates the actual work to the device object.
    """
    def __init__(self, device: Device):
        self.device = device  # <--- THE BRIDGE

    def toggle_power(self):
        if self.device.is_enabled():
            self.device.disable()
        else:
            self.device.enable()

    def volume_down(self):
        print("Remote: Volume Down")
        self.device.set_volume(0) # Logic simplified

# --- Refined Abstraction ---
class AdvancedRemoteControl(RemoteControl):
    """
    Extends the Remote functionality without changing the Device classes.
    """
    def mute(self):
        print("Remote: Mute Button Pressed")
        self.device.set_volume(0)

#### CLIENT CODE

In [5]:
def main():
    tv = TV()
    radio = Radio()

    # We can pair ANY remote with ANY device
    remote_tv = RemoteControl(tv)
    remote_tv.toggle_power()

    print("-" * 20)

    # Advanced remote works with Radio too
    advanced_remote_radio = AdvancedRemoteControl(radio)
    advanced_remote_radio.toggle_power()
    advanced_remote_radio.mute()

if __name__ == "__main__":
    main()

TV: Turned ON
--------------------
Radio: Turned ON
Remote: Mute Button Pressed
Radio: Volume set to 0


## The Pythonic Way

In Python, we can simplify this using Protocols (Duck Typing). We don't need rigid inheritance for the implementation layer (TV doesn't need to inherit from Device). As long as it has the methods, the bridge works.

We also use Composition more naturally.

#### PROTOCOL (Implicit Interface)

In [6]:
from typing import Protocol

class Device(Protocol):
    def turn_on(self) -> None: ...
    def turn_off(self) -> None: ...
    def set_channel(self, channel: int) -> None: ...

#### CONCRETE DEVICES (No Inheritance needed)

In [7]:
class TV:
    def turn_on(self): print("TV: ON")
    def turn_off(self): print("TV: OFF")
    def set_channel(self, channel): print(f"TV: Channel {channel}")

class Radio:
    def turn_on(self): print("Radio: ON")
    def turn_off(self): print("Radio: OFF")
    def set_channel(self, channel): print(f"Radio: Frequency {channel}")

#### THE BRIDGE (The Remotes)

In [8]:
class BaseRemote:
    def __init__(self, device: Device):
        self.device = device # The Bridge

    def power(self):
        print("Remote: Power button pressed.")
        self.device.turn_on() # Simple delegation

class AdvancedRemote(BaseRemote):
    def mute(self):
        print("Remote: Mute.")
        # We can add logic here without touching the TV class
        self.device.set_channel(0)

#### CLIENT CODE

In [9]:
def main():
    # Mix and Match
    my_tv = TV()
    my_radio = Radio()

    # Bridge: Connect Advanced Remote -> TV
    remote = AdvancedRemote(my_tv)
    remote.power()
    remote.mute()

    print("--- Switching Device ---")

    # Bridge: Reuse SAME Remote logic -> Radio
    # We swapped the implementation at runtime!
    remote.device = my_radio
    remote.power()

if __name__ == "__main__":
    main()

Remote: Power button pressed.
TV: ON
Remote: Mute.
TV: Channel 0
--- Switching Device ---
Remote: Power button pressed.
Radio: ON


| Key Differences | Feature      | Classic OOP                                   | Pythonic                                        |
|-----------------|--------------|-----------------------------------------------|-------------------------------------------------|
| Device Layer    | Implementation | Inherits from an abstract `Device` class.     | Uses `Protocol` (duck typing). TV is standalone. |
| Coupling        | Dependency   | Tighter coupling via inheritance.             | Loose coupling via structural typing.           |
| Flexibility     | Design       | High flexibility, but requires more setup code. | Very high flexibility with minimal boilerplate.  |


### Summary

The Bridge pattern is essentially Composition over Inheritance taken to the architectural level. Instead of class TvRemote(Remote), you have class Remote which has a TV.